# Manifest Package Operator Tutorial

This notebook only creates and validates destination-independent manifest packages. It does not publish to ADLS or submit Fabric work.

The operator workflow is split into two tools:

1. **This notebook** discovers and validates videos, calculates hashes, and writes the package locally or in a Fabric Lakehouse.
2. **The companion [backfill operator notebook](explicit_backfill_operator_tutorial.ipynb)** previews the package, then publishes and registers work only when explicitly confirmed. It uses the separate `people_counter.backfill_operator` command. See also the [operator guide](../docs/explicit-backfill.md).

> **Safety:** running every cell only prepares package files. Fabric mode reads the mounted source shortcut and writes to the Lakehouse using the notebook's access. It never publishes videos, registers work, or submits jobs.

## 1. Roles and prerequisites

The only data input is the tutorial's source videos. In both local and Fabric modes, this notebook generates the same example camera catalog and timestamp overrides. No catalog upload or manual metadata inventory is required. The sample calibration and timestamps describe the tutorial clips, not arbitrary production footage.

Local prerequisites:

- Python 3.10 or newer and `uv`;
- this repository and its installed environment (run `uv sync --extra experiments` before opening the notebook, then select its `.venv` kernel);
- FFmpeg's `ffprobe` on `PATH`.

Fabric prerequisites:

- Import this notebook as a PySpark notebook, attach the default Lakehouse and a published Environment containing the compatible `people-counter` SDK, and ensure `ffprobe` is on `PATH`. No repository checkout, `uv`, or Azure CLI login is required.
- Make the tutorial clips readable below `VIDEO_ROOT` at the relative folder configured by `PARTITION_PREFIX`. For example, create a shortcut named `tutorial-videos` inside `Files/backfill/source-videos` and set `PARTITION_PREFIX = "tutorial-videos/"`, targeting the folder containing `people_crossing.mp4`, `subway.mp4`, and `three_people_walking.mp4`. The notebook identity needs source read access and write access to the package/configuration folders.
- The camera catalog and exception inventory are generated under `Files/backfill/config`, exactly as they are generated locally. These tutorial files are rewritten when their cells run; use a dedicated tutorial output folder, not production camera configuration.
- Keep the package and configuration folders outside the read-only source shortcut. Use Python-accessible mounted paths (`/lakehouse/default/Files/...`), not `abfss://` URLs with `Path`.

**If Fabric reports `FileNotFoundError: ffprobe`:** open the attached Environment, select **Libraries > External repositories > Add library from public repository**, choose **Conda**, and add `ffmpeg` at an approved compatible version. Save and publish the Environment, then stop the notebook session, start a new one attached to that Environment, and rerun from step 1. The Conda FFmpeg distribution supplies the `ffprobe` executable; `pip install ffmpeg` or `ffmpeg-python` does not. Verify in the new session with `import shutil; print(shutil.which("ffprobe"))` and `import subprocess; subprocess.run(["ffprobe", "-version"], check=True)`. See [Fabric environment library management](https://learn.microsoft.com/en-us/fabric/data-engineering/environment-manage-library).

Set `FABRIC_MODE = True` below to prepare directly in Fabric. Configure `PARTITION_PREFIX` in the branch you use: it can be any non-empty relative folder beneath `VIDEO_ROOT`, including nested folders, with or without a trailing slash. It defaults to `samples/` in both modes. The notebook normalizes the prefix and uses it for discovery, generated camera metadata, and the timestamp override; all other tutorial metadata stays the same. After changing it, rerun setup and the subsequent cells to regenerate the package.

`PACKAGE_DIR` and `VIDEO_ROOT` default to the companion backfill notebook's Fabric paths. If you change either, use the same value in both notebooks. Preserve the chosen video-relative layout for the backfill notebook; changing the prefix changes manifest asset identities.

Fabric preparation keeps its SQLite checkpoint in temporary driver-local storage, not OneLake. It is only a preparation cache: after a session restart, rerun preparation from the unchanged inputs. The completed package remains in the Lakehouse.

Do not run `uv sync` or environment-changing `uv run` commands against the notebook's environment while its kernel is running. If the environment has been replaced, restart the kernel, select the current `.venv`, and rerun from step 1.

Never place credentials in the catalog, exception inventory, package, notebook, command history, or logs.

In [ ]:
from pathlib import Path
import csv
import hashlib
import json
import shutil
import subprocess
import sys
from tempfile import TemporaryDirectory

from people_counter.manifest import normalize_relative_path
from people_counter.manifest_publisher import discover_videos

FABRIC_MODE = False


def find_repo_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "samples").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from inside the people-counter repository")


if FABRIC_MODE:
    LAKEHOUSE_ROOT = Path("/lakehouse/default")
    VIDEO_ROOT = LAKEHOUSE_ROOT / "Files/backfill/source-videos"
    PACKAGE_DIR = LAKEHOUSE_ROOT / "Files/backfill/manifest-package-tutorial"
    INPUT_DIR = LAKEHOUSE_ROOT / "Files/backfill/config"
    PARTITION_PREFIX = "samples/"
else:
    REPO_ROOT = find_repo_root()
    VIDEO_ROOT = REPO_ROOT
    INPUT_DIR = REPO_ROOT / "samples" / "manifest-package-tutorial-inputs"
    PACKAGE_DIR = REPO_ROOT / "samples" / "manifest-package-tutorial"
    PARTITION_PREFIX = "samples/"

PARTITION_PREFIX = normalize_relative_path(PARTITION_PREFIX, directory=True)
MAX_FILES = 10
CATALOG_PATH = INPUT_DIR / "camera_catalog.csv"
EXCEPTION_INVENTORY_PATH = INPUT_DIR / "video-inventory-exceptions.csv"

if not VIDEO_ROOT.is_dir():
    raise FileNotFoundError(f"Video root is missing or inaccessible; check the Lakehouse shortcut: {VIDEO_ROOT}")
if shutil.which("ffprobe") is None:
    raise RuntimeError(
        "ffprobe is missing from PATH. In Fabric, add the Conda ffmpeg library "
        "to the attached Environment, publish it, and restart the notebook session. "
        "Locally, install FFmpeg. Python ffmpeg wrappers do not supply ffprobe."
    )
INPUT_DIR.mkdir(parents=True, exist_ok=True)
if FABRIC_MODE:
    PREPARATION_STATE = TemporaryDirectory(prefix="pc-manifest-prepare-")
    CHECKPOINT_PATH = Path(PREPARATION_STATE.name) / "preparation.sqlite3"
else:
    CHECKPOINT_PATH = PACKAGE_DIR / "preparation.sqlite3"

print(f"Video root: {VIDEO_ROOT}")
print(f"Package output: {PACKAGE_DIR}")
print(f"Camera catalog: {CATALOG_PATH}")

## 2. Create the camera metadata catalog

This cell generates the same sample camera catalog in both modes. Create one effective-dated row for each camera configuration, not one row per video. A row defines:

- the source path prefix used to associate videos with a camera;
- stable camera and location identifiers;
- the IANA camera timezone;
- expected frame dimensions;
- the directed counting line;
- how capture time is resolved; and
- the effective UTC interval for that configuration.

Use `capture_time_source=auto` in most cases. Resolution order is:

1. an optional exception-inventory override;
2. embedded stream/container `creation_time`;
3. the configured filename regular expression.

The counting line direction matters: reversing its endpoints reverses `in` and `out`. Coordinates must fit within the declared frame dimensions.

In [ ]:
catalog_fields = [
    "catalog_version", "source_path_prefix", "camera_id", "location_id",
    "camera_timezone", "frame_width", "frame_height",
    "counting_line_x1", "counting_line_y1", "counting_line_x2", "counting_line_y2",
    "capture_time_source", "capture_time_regex", "capture_time_format",
    "effective_from_utc", "effective_to_utc",
]

# The sample clips have different dimensions and effective capture dates.
catalog_rows = [
    {
        "catalog_version": 1, "source_path_prefix": PARTITION_PREFIX,
        "camera_id": "sample-camera", "location_id": "sample-location",
        "camera_timezone": "UTC", "frame_width": 3626, "frame_height": 1994,
        "counting_line_x1": 0, "counting_line_y1": 997,
        "counting_line_x2": 3625, "counting_line_y2": 997,
        "capture_time_source": "auto", "capture_time_regex": "",
        "capture_time_format": "", "effective_from_utc": "2023-01-01T00:00:00Z",
        "effective_to_utc": "2023-09-28T00:00:00Z",
    },
    {
        "catalog_version": 1, "source_path_prefix": PARTITION_PREFIX,
        "camera_id": "sample-camera", "location_id": "sample-location",
        "camera_timezone": "UTC", "frame_width": 3840, "frame_height": 2160,
        "counting_line_x1": 0, "counting_line_y1": 1080,
        "counting_line_x2": 3839, "counting_line_y2": 1080,
        "capture_time_source": "auto", "capture_time_regex": "",
        "capture_time_format": "", "effective_from_utc": "2023-09-28T00:00:00Z",
        "effective_to_utc": "2023-10-01T00:00:00Z",
    },
    {
        "catalog_version": 1, "source_path_prefix": PARTITION_PREFIX,
        "camera_id": "sample-camera", "location_id": "sample-location",
        "camera_timezone": "UTC", "frame_width": 1920, "frame_height": 1080,
        "counting_line_x1": 0, "counting_line_y1": 540,
        "counting_line_x2": 1919, "counting_line_y2": 540,
        "capture_time_source": "auto", "capture_time_regex": "",
        "capture_time_format": "", "effective_from_utc": "2023-10-01T00:00:00Z",
        "effective_to_utc": "",
    },
]

with CATALOG_PATH.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=catalog_fields)
    writer.writeheader()
    writer.writerows(catalog_rows)

print(CATALOG_PATH.read_text(encoding="utf-8"))

## 3. Preview source-video metadata

Preparation performs this inspection automatically. The preview below helps an operator understand which timestamps can be derived without manual input.

Do not use filesystem modification time or upload time as recording time. If embedded metadata and filename rules cannot provide a trustworthy timestamp, preparation rejects that video with an actionable reason code.

In [ ]:
video_paths = discover_videos(VIDEO_ROOT, PARTITION_PREFIX, MAX_FILES)

for video_path in video_paths:
    completed = subprocess.run(
        [
            "ffprobe", "-v", "error", "-select_streams", "v:0",
            "-show_entries",
            "stream=width,height,duration:stream_tags=creation_time:format=duration:format_tags=creation_time",
            "-of", "json", str(video_path),
        ],
        cwd=VIDEO_ROOT,
        check=True,
        capture_output=True,
        text=True,
    )
    metadata = json.loads(completed.stdout)
    stream = metadata["streams"][0]
    embedded_time = stream.get("tags", {}).get("creation_time")
    duration = stream.get("duration") or metadata.get("format", {}).get("duration")
    print(
        video_path.relative_to(VIDEO_ROOT),
        f"{stream['width']}x{stream['height']}",
        f"duration={duration}",
        f"creation_time={embedded_time}",
    )

## 4. Run preparation without exception overrides

The first pass intentionally omits `--inventory`. Two sample videos have embedded timestamps; one does not. Exit code `2` means preparation completed but at least one video was rejected.

An incomplete package is never publishable. Review grouped totals in `summary.json` and individual remediation in `rejection-report.csv`.

In [ ]:
prepare_base = [
    sys.executable, "-c",
    "from people_counter.manifest_publisher import prepare_main; raise SystemExit(prepare_main())",
    "--catalog", str(CATALOG_PATH),
    "--video-root", str(VIDEO_ROOT),
    "--partition-prefix", PARTITION_PREFIX,
    "--output-dir", str(PACKAGE_DIR),
    "--checkpoint", str(CHECKPOINT_PATH),
    "--max-files", str(MAX_FILES),
]

first_pass = subprocess.run(
    prepare_base,
    cwd=VIDEO_ROOT,
    check=False,
    capture_output=True,
    text=True,
)
print(first_pass.stdout.strip())
if first_pass.stderr.strip():
    print(first_pass.stderr.strip())
if first_pass.returncode not in {0, 2}:
    raise RuntimeError(f"Preparation failed with exit code {first_pass.returncode}")
print(f"Exit code: {first_pass.returncode}")

In [ ]:
summary = json.loads((PACKAGE_DIR / "summary.json").read_text(encoding="utf-8"))
with (PACKAGE_DIR / "rejection-report.csv").open(newline="", encoding="utf-8") as handle:
    rejections = list(csv.DictReader(handle))

print(json.dumps(summary, indent=2, sort_keys=True))
print("\nRejections:")
for rejection in rejections:
    print(
        f"- {rejection['video_relative_path']}: {rejection['reason_code']}\n"
        f"  Explanation: {rejection['explanation']}\n"
        f"  Suggested action: {rejection['suggested_action']}"
    )

## 5. Add only exceptional capture-time overrides

Do not manually inventory all videos. The optional inventory exists only for files whose trustworthy capture time cannot be derived by code.

The timestamp below is the same tutorial value in both modes. This cell generates the exception inventory automatically; no upload is required. In production, obtain timestamps from an authoritative camera or source-system record and include an explicit UTC `Z` or numeric offset.

In [ ]:
exception_rows = [
    {
        "video_relative_path": f"{PARTITION_PREFIX}three_people_walking.mp4",
        "captured_at_utc": "2023-10-01T01:00:00Z",
    }
]

with EXCEPTION_INVENTORY_PATH.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=["video_relative_path", "captured_at_utc"],
    )
    writer.writeheader()
    writer.writerows(exception_rows)

print(EXCEPTION_INVENTORY_PATH.read_text(encoding="utf-8"))

In [ ]:
complete_pass = subprocess.run(
    [
        *prepare_base,
        "--inventory", str(EXCEPTION_INVENTORY_PATH),
    ],
    cwd=VIDEO_ROOT,
    check=True,
    capture_output=True,
    text=True,
)
print(complete_pass.stdout.strip())

## 6. Inspect the completed manifest package

`manifest-package.json` is the authoritative checksum index. Prepared manifests intentionally omit `video_uri` and `source_etag`; publication adds those storage-dependent values later.

Identity is deterministic:

```text
asset_id      = SHA256(normalized source-relative path)
asset_version = SHA256(video bytes)
```

Changing one byte creates a new asset version. Correcting bytes at the same relative path retains the logical asset ID.

In [ ]:
package_index = json.loads(
    (PACKAGE_DIR / "manifest-package.json").read_text(encoding="utf-8")
)
package_summary = json.loads(
    (PACKAGE_DIR / "summary.json").read_text(encoding="utf-8")
)
with (PACKAGE_DIR / "generated-video-inventory.csv").open(
    newline="", encoding="utf-8"
) as handle:
    generated_inventory = list(csv.DictReader(handle))

print("Package files:")
for path in sorted(PACKAGE_DIR.rglob("*")):
    if path.is_file():
        print("-", path.relative_to(PACKAGE_DIR))

print("\nIndex summary:")
print(json.dumps({
    key: package_index[key]
    for key in ("manifest_package_version", "complete", "discovered", "prepared", "rejected")
}, indent=2))
assert package_index["complete"] is True
assert package_index["prepared"] == len(generated_inventory) > 0
assert not list(PACKAGE_DIR.rglob("*.mp4"))

sample_entry = package_index["entries"][0]
sample_prepared_path = PACKAGE_DIR / sample_entry["prepared_manifest_path"]
sample_prepared = json.loads(sample_prepared_path.read_text(encoding="utf-8"))
assert "video_uri" not in sample_prepared
assert "source_etag" not in sample_prepared
print("\nExample prepared manifest:")
print(json.dumps(sample_prepared, indent=2, sort_keys=True))

In [ ]:
for row in generated_inventory:
    relative_path = row["video_relative_path"]
    video_path = VIDEO_ROOT.joinpath(*relative_path.split("/"))
    expected_asset_id = hashlib.sha256(relative_path.encode("utf-8")).hexdigest()
    video_digest = hashlib.sha256()
    with video_path.open("rb") as source:
        while chunk := source.read(8 * 1024 * 1024):
            video_digest.update(chunk)
    assert row["asset_id"] == expected_asset_id
    assert row["asset_version"] == video_digest.hexdigest()
    assert row["sha256"] == row["asset_version"]
    print(relative_path, row["asset_version"])

print("All generated identities match the source paths and exact video bytes.")

## 7. Validate the completed package offline

This validation reads the package and rehashes every referenced video before any publication client is created. In Fabric, mounted reads still use the notebook's Lakehouse/shortcut access; no publication or job submission API is called. It catches edited package files, missing videos, changed bytes, unsupported versions, duplicate identities, inconsistent counts, and incomplete preparation.

In [ ]:
from people_counter.manifest_publisher import (
    ManifestPackagePublishConfig,
    validate_manifest_package,
)

validation_config = ManifestPackagePublishConfig(
    manifest_package_dir=PACKAGE_DIR,
    video_root=VIDEO_ROOT,
    storage_account="validation-only",
    filesystem="validation-only",
    staging_prefix="staging",
    incoming_prefix="incoming",
    checkpoint_path=PACKAGE_DIR / "unused-publication.sqlite3",
    rejection_report_path=PACKAGE_DIR / "unused-publication-rejections.csv",
    summary_report_path=PACKAGE_DIR / "unused-publication-summary.json",
    chunk_size=8 * 1024 * 1024,
)

validated_package = validate_manifest_package(validation_config)
print(f"Validated {len(validated_package.plans)} videos without publication or job submission.")

## 8. Retain or transfer the package

Publication may happen immediately or later. The publication operator needs:

1. the unchanged source-video directory tree; and
2. the complete manifest package directory.

The absolute path may change, but relative paths under `--video-root` must remain the same. SHA-256 integrity is authoritative: publication rehashes the video files and refuses changed or missing bytes.

Do not edit prepared JSON files or `manifest-package.json`. Correct source/catalog data and rerun preparation instead.

## 9. Package complete: hand off for submission

Stop here. Retain `PACKAGE_DIR` and the unchanged source tree under `VIDEO_ROOT`. The notebook has not uploaded or registered anything.

Import the companion [Explicit Backfill Operator Tutorial](explicit_backfill_operator_tutorial.ipynb) into Fabric and set `FABRIC_MODE = True` there too. With the default paths, it reads this package directly from `Files/backfill/manifest-package-tutorial` and the same shortcut at `Files/backfill/source-videos`; no copying or publication is needed between notebooks. If you customized `PACKAGE_DIR` or `VIDEO_ROOT`, copy those exact settings to the second notebook. Configure its approved model directory and administrator-reviewed deployment bindings before preview or submission.

If preparation ran locally, transfer the complete package and map the unchanged source-video tree to Lakehouse folders, preserving video-relative paths. The backfill notebook generates its catalog from real model bytes, previews, and triggers backfill only with explicit confirmation using the notebook identity. The [explicit backfill guide](../docs/explicit-backfill.md) also covers local CLI operation and administrator setup.

Do not create registration JSON, calculate partition hashes, or submit SJD jobs by hand. The separate command performs those operations and retains retry evidence outside the package.

## 10. Preparation troubleshooting

| Reason code | Action |
|---|---|
| `CAPTURE_TIME_MISSING` | Repair embedded metadata, add a filename timestamp rule, or add only affected videos to the exception inventory |
| `CAPTURE_TIME_FILENAME_MISMATCH` | Correct the catalog regex/format or supply an authoritative override |
| `NO_CAMERA_MATCH` | Add or correct the catalog `source_path_prefix` |
| `CAMERA_EFFECTIVE_RANGE_MISMATCH` | Correct the timestamp or effective-dated catalog rows |
| `VIDEO_DIMENSION_MISMATCH` | Correct catalog dimensions/counting line or add another effective-dated configuration |
| `VIDEO_PROBE_FAILED` | Validate, repair, or remux the source video |
| `MANIFEST_PACKAGE_INCOMPLETE` | Resolve all preparation rejections and regenerate the package |
| `MANIFEST_PACKAGE_CHECKSUM_MISMATCH` | Recopy or regenerate the package; never edit prepared files |
| `MANIFEST_PACKAGE_VIDEO_MISMATCH` | Restore the exact source bytes or regenerate from the current video tree |

For large backfills, use the grouped rejection counts in `summary.json` to fix the largest issue classes first instead of reviewing console logs one by one.